# Ayudantía 2 — SQL práctico con SQLite
### Clases 3 y 4 — IIC2413

En esta ayudantía vamos a:

1. Instalar / verificar lo necesario para trabajar con SQLite desde Python.
2. Entender el código de "setup": cómo nos conectamos a una base de datos, qué es un cursor, y cómo se crean tablas e insertan datos.
3. Resolver ejercicios prácticos de SQL.
4. Practicar el patrón de **función parametrizada con cursor** (usando `?`) en algunos ejercicios.

> Esta ayudantía usa un dataset propio (una tienda de videojuegos).

## Parte 0 — Instalación

`sqlite3` viene **incluido en la librería estándar de Python**: no necesitas instalar nada extra, basta con `import sqlite3`.

**(Opcional)** Si quieres explorar la base de datos de forma gráfica o desde la línea de comandos:

- [DB Browser for SQLite](https://sqlitebrowser.org/) (GUI)
- Cliente `sqlite3` por línea de comandos:
  - Linux (Debian/Ubuntu): `sudo apt install sqlite3`
  - Linux (Fedora): `sudo dnf install sqlite`
  - macOS: `brew install sqlite`
  - Windows: `choco install sqlite`

Nada de esto es obligatorio: con Python y `sqlite3` ya tienes todo lo necesario para la ayudantía y la tarea.

In [ ]:
import sqlite3

## Parte 1 — Conexión y cursor

Para hablar con una base de datos SQLite desde Python necesitamos dos conceptos clave:

- **Conexión (`Connection`)**: es el enlace hacia el archivo de base de datos (`.db`). Se abre con `sqlite3.connect(path)` y se cierra con `.close()`. Los cambios hechos con DDL/DML (`CREATE`, `INSERT`, `UPDATE`, `DELETE`) solo quedan guardados de forma permanente si haces `conn.commit()`.
- **Cursor (`Cursor`)**: es el objeto que efectivamente **ejecuta** sentencias SQL sobre una conexión (`cursor.execute(query)`) y te permite **leer los resultados** fila por fila (`cursor.fetchone()`, `cursor.fetchall()`).

## Parte 2 — Nuestra base de datos: una tienda de videojuegos

Vamos a trabajar con 3 tablas, con todos sus atributos en español:

```
desarrolladores(id_desarrollador, nombre, pais)                                            ar = 3
juegos(id_juego, titulo, id_desarrollador, genero, plataforma, precio, anio_lanzamiento)    ar = 7
compras(id_compra, id_juego, nombre_comprador, fecha_compra)                                ar = 4
```

donde `juegos.id_desarrollador` referencia a `desarrolladores.id_desarrollador` (cada juego pertenece a un desarrollador), y `compras.id_juego` referencia a `juegos.id_juego` (cada compra corresponde a un juego).

Los datos están **hardcodeados** en el notebook (no hay `.csv` externo): es autocontenido y no depende de archivos externos.

In [ ]:
desarrolladores = [
    (1, "CD Projekt Red", "Polonia"),
    (2, "FromSoftware", "Japón"),
    (3, "Naughty Dog", "Estados Unidos"),
    (4, "Mojang Studios", "Suecia"),
    (5, "Valve", "Estados Unidos"),
    (6, "Nintendo", "Japón"),
    (7, "Rockstar Games", "Estados Unidos"),
    (8, "Larian Studios", "Bélgica"),
    (9, "ConcernedApe", "Estados Unidos"),
    (10, "Supergiant Games", "Estados Unidos"),
    (11, "Santa Monica Studio", "Estados Unidos"),
    (12, "Team Cherry", "Australia"),
    (13, "Maddy Makes Games", "Canadá"),
]

juegos = [
    (1, "The Witcher 3", 1, "RPG", "PC", 29.99, 2015),
    (2, "Cyberpunk 2077", 1, "RPG", "PC", 49.99, 2020),
    (3, "Elden Ring", 2, "RPG", "PS5", 59.99, 2022),
    (4, "Dark Souls III", 2, "RPG", "PC", 39.99, 2016),
    (5, "The Last of Us Part I", 3, "Acción", "PS5", 69.99, 2022),
    (6, "Uncharted 4", 3, "Acción", "PS5", 39.99, 2016),
    (7, "Minecraft", 4, "Sandbox", "PC", 26.99, 2011),
    (8, "Half-Life 2", 5, "Acción", "PC", 9.99, 2004),
    (9, "Portal 2", 5, "Puzzle", "PC", 9.99, 2011),
    (10, "Dota 2", 5, "Estrategia", "PC", 0.00, 2013),
    (11, "The Legend of Zelda: BOTW", 6, "Aventura", "Switch", 59.99, 2017),
    (12, "Super Mario Odyssey", 6, "Plataformas", "Switch", 59.99, 2017),
    (13, "Red Dead Redemption 2", 7, "Acción", "PS5", 59.99, 2018),
    (14, "Grand Theft Auto V", 7, "Acción", "PC", 29.99, 2013),
    (15, "Baldur's Gate 3", 8, "RPG", "PC", 59.99, 2023),
    (16, "Divinity: Original Sin 2", 8, "RPG", "PC", 44.99, 2017),
    (17, "Stardew Valley", 9, "Simulación", "PC", 14.99, 2016),
    (18, "Hades", 10, "Acción", "PC", 24.99, 2020),
    (19, "Pyre", 10, "RPG", "PC", 19.99, 2017),
    (20, "Bastion", 10, "Acción", "PC", 14.99, 2011),
    (21, "Mario Kart 8 Deluxe", 6, "Carreras", "Switch", 59.99, 2017),
    (22, "Animal Crossing: New Horizons", 6, "Simulación", "Switch", 59.99, 2020),
    (23, "God of War Ragnarök", 11, "Acción", "PS5", 69.99, 2022),
    (24, "Hollow Knight", 12, "Metroidvania", "PC", 14.99, 2017),
    (25, "Celeste", 13, "Plataformas", "PC", 19.99, 2018),
]

compras = [
    (1, 1, "Ana Torres", "2023-01-15"),
    (2, 3, "Ana Torres", "2023-02-10"),
    (3, 11, "Ana Torres", "2023-03-05"),
    (4, 2, "Bruno Silva", "2023-01-20"),
    (5, 5, "Bruno Silva", "2023-04-12"),
    (6, 13, "Bruno Silva", "2023-05-01"),
    (7, 7, "Camila Rojas", "2023-02-14"),
    (8, 17, "Camila Rojas", "2023-06-18"),
    (9, 21, "Camila Rojas", "2023-07-22"),
    (10, 3, "Diego Fuentes", "2023-03-30"),
    (11, 4, "Diego Fuentes", "2023-08-09"),
    (12, 15, "Diego Fuentes", "2023-09-14"),
    (13, 1, "Elena Paredes", "2023-04-02"),
    (14, 12, "Elena Paredes", "2023-05-19"),
    (15, 22, "Elena Paredes", "2023-06-25"),
    (16, 8, "Felipe Muñoz", "2023-01-08"),
    (17, 9, "Felipe Muñoz", "2023-02-28"),
    (18, 18, "Felipe Muñoz", "2023-10-05"),
    (19, 6, "Gabriela Soto", "2023-03-11"),
    (20, 13, "Gabriela Soto", "2023-04-22"),
    (21, 23, "Gabriela Soto", "2023-11-01"),
    (22, 2, "Hugo Vargas", "2023-05-06"),
    (23, 16, "Hugo Vargas", "2023-06-30"),
    (24, 15, "Ignacia Bravo", "2023-07-15"),
    (25, 11, "Ignacia Bravo", "2023-08-20"),
    (26, 20, "Javiera Núñez", "2023-02-02"),
    (27, 14, "Javiera Núñez", "2023-09-09"),
    (28, 1, "Kevin Castro", "2023-01-30"),
    (29, 5, "Kevin Castro", "2023-10-14"),
    (30, 12, "Laura Méndez", "2023-11-20"),
    (31, 21, "Laura Méndez", "2023-12-01"),
    (32, 3, "Manuel Díaz", "2024-01-05"),
    (33, 23, "Manuel Díaz", "2024-01-12"),
]

### Creando las tablas (DDL) e insertando los datos (DML)

Usamos `cursor.executescript(...)` para correr varias sentencias DDL de una vez (`DROP TABLE` + `CREATE TABLE`), y `cursor.executemany(...)` para insertar **todas** las filas de una lista con una sola sentencia `INSERT` parametrizada (cada `?` se reemplaza por un valor de la tupla correspondiente).

In [ ]:
def setup_tienda_db(db_path):
    """Crea (desde cero) la base de datos de la tienda de videojuegos."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.executescript("""
        DROP TABLE IF EXISTS compras;
        DROP TABLE IF EXISTS juegos;
        DROP TABLE IF EXISTS desarrolladores;

        CREATE TABLE desarrolladores (
            id_desarrollador INTEGER,
            nombre TEXT NOT NULL,
            pais TEXT
        );

        CREATE TABLE juegos (
            id_juego INTEGER,
            titulo TEXT NOT NULL,
            id_desarrollador INTEGER,
            genero TEXT,
            plataforma TEXT,
            precio REAL,
            anio_lanzamiento INTEGER
        );

        CREATE TABLE compras (
            id_compra INTEGER,
            id_juego INTEGER,
            nombre_comprador TEXT NOT NULL,
            fecha_compra TEXT
        );
    """)

    cursor.executemany("INSERT INTO desarrolladores VALUES (?, ?, ?);", desarrolladores)
    cursor.executemany("INSERT INTO juegos VALUES (?, ?, ?, ?, ?, ?, ?);", juegos)
    cursor.executemany("INSERT INTO compras VALUES (?, ?, ?, ?);", compras)

    conn.commit()
    conn.close()

setup_tienda_db("tienda.db")

Probemos que quedó todo bien:

In [ ]:
conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute("SELECT * FROM juegos")
resultados = cursor.fetchall()
conn.close()

resultados

---
## Bloque A — Proyección, selección y `DISTINCT`


### Ejercicio A1 (resuelto)
Muestra el título y el precio de todos los juegos de género `'RPG'`.

In [ ]:
a1 = """
    SELECT titulo, precio
    FROM juegos
    WHERE genero = 'RPG';
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(a1)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** Aplicamos una proyección (`titulo, precio`) sobre una selección simple (`genero = 'RPG'`), el orden `SELECT → FROM → WHERE` corresponde a π(σ(R)).

### Ejercicio A2 (resuelto — función parametrizada · Ejercicio 1 en las slides)
Escribe una función `juegos_por_plataforma_y_precio(db_path, plataforma, precio_maximo)` que retorne el título de los juegos de una plataforma dada que cuesten menos que un precio máximo dado. Pruébala con `'PC'` y `20`.

A diferencia de A1, acá no dejamos los valores fijos en el `WHERE`: los recibimos como parámetros de la función y los pasamos con un placeholder `?`.

In [ ]:
def juegos_por_plataforma_y_precio(db_path, plataforma, precio_maximo):
    """Retorna (titulo,) de los juegos de una plataforma dada con precio menor a precio_maximo."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
        SELECT titulo
        FROM juegos
        WHERE plataforma = ? AND precio < ?;
    """
    cursor.execute(query, (plataforma, precio_maximo))
    resultados = cursor.fetchall()
    conn.close()

    return resultados

juegos_por_plataforma_y_precio("tienda.db", "PC", 20)

**Explicación:** cada `?` se reemplaza, en orden, por un valor de la tupla `(plataforma, precio_maximo)`. Así, la misma función sirve para cualquier plataforma o precio (no solo `'PC'` y `20`) sin tocar el string SQL.

### Ejercicio A3
Muestra, **sin duplicados**, todos los géneros (`genero`) que existen en la tabla `juegos`.

*Pista: usa `DISTINCT`*

In [ ]:
a3 = """
    -- escribe tu query aquí
"""

# conn = sqlite3.connect("tienda.db")
# cursor = conn.cursor()
# cursor.execute(a3)
# resultados = cursor.fetchall()
# conn.close()
# resultados

#### Solución A3

In [ ]:
a3 = """
    SELECT DISTINCT genero
    FROM juegos;
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(a3)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** `DISTINCT` elimina filas repetidas *después* de la proyección. Sin él, cada juego aportaría su género aunque varios juegos compartan el mismo.

### Ejercicio A4
Muestra el título (con el alias `Juego`) y el año de lanzamiento (con el alias `Anio`) de los juegos lanzados **antes** del año 2015.

*Pista: usa `AS` para renombrar columnas en el `SELECT`*

In [ ]:
a4 = """
    -- escribe tu query aquí
"""

# conn = sqlite3.connect("tienda.db")
# cursor = conn.cursor()
# cursor.execute(a4)
# resultados = cursor.fetchall()
# conn.close()
# resultados

#### Solución A4

In [ ]:
a4 = """
    SELECT titulo AS Juego, anio_lanzamiento AS Anio
    FROM juegos
    WHERE anio_lanzamiento < 2015;
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(a4)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** Los alias con `AS` solo cambian el nombre de la columna en el resultado mostrado: no afectan los datos originales ni se pueden usar en el mismo `WHERE` que los define.

---
## Bloque B — Producto cruz y JOINs

Antes de los joins "de verdad", recordemos el peligro del producto cruz (`FROM t1, t2` sin `WHERE`): junta **cada fila con cada fila**, generando combinaciones sin sentido.

In [ ]:
demo_cruz = """
    SELECT j.titulo, d.nombre
    FROM juegos AS j, desarrolladores AS d;
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(demo_cruz)
resultado = cursor.fetchall()
conn.close()

print(f"Filas generadas: {len(resultado)}  (25 juegos x 13 desarrolladores = 325, la mayoría sin sentido)")
resultado[:5]

La forma más usada en la práctica para hacer un join es el **estilo `WHERE`**: producto cruz + una condición de igualdad entre las columnas que conectan las tablas.

### Ejercicio B1 (resuelto — Ejercicio 4 en las slides)
Muestra el título del juego y el país de su desarrollador, para los juegos de género `'RPG'`.

In [ ]:
b1 = """
    SELECT j.titulo, d.pais
    FROM juegos AS j, desarrolladores AS d
    WHERE j.id_desarrollador = d.id_desarrollador
      AND j.genero = 'RPG';
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(b1)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** Este es el join \"estilo WHERE\": producto cruz (`FROM j, d`) más una condición de igualdad entre las columnas que conectan las tablas. Es equivalente a escribir `FROM juegos AS j JOIN desarrolladores AS d ON j.id_desarrollador = d.id_desarrollador`. Ambas formas son intercambiables.

### Ejercicio B2 (resuelto — función parametrizada · Ejercicio 5 en las slides)
Escribe una función `compras_de(db_path, nombre_comprador)` que retorne el título del juego y la fecha de compra de todas las compras de un comprador dado. Pruébala con `'Gabriela Soto'`.

In [ ]:
def compras_de(db_path, nombre_comprador):
    """Retorna (titulo, fecha_compra) de las compras de un comprador dado."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
        SELECT j.titulo, c.fecha_compra
        FROM compras AS c
        JOIN juegos AS j ON c.id_juego = j.id_juego
        WHERE c.nombre_comprador = ?;
    """
    cursor.execute(query, (nombre_comprador,))
    resultados = cursor.fetchall()
    conn.close()

    return resultados

compras_de("tienda.db", "Gabriela Soto")

**Explicación:** un solo parámetro, un solo `?`. Ojo con la coma en `(nombre_comprador,)`: sin ella, Python no lo interpreta como una tupla de un elemento.

### Ejercicio B3
Usando `JOIN ... USING`, muestra el título del juego y el nombre de su desarrollador, solo para los juegos de la plataforma `'Switch'`.

*Pista: `USING` sirve cuando la columna de join se llama igual en ambas tablas (`id_desarrollador`)*

In [ ]:
b3 = """
    -- escribe tu query aquí
"""

# conn = sqlite3.connect("tienda.db")
# cursor = conn.cursor()
# cursor.execute(b3)
# resultados = cursor.fetchall()
# conn.close()
# resultados

#### Solución B3

In [ ]:
b3 = """
    SELECT titulo, nombre
    FROM juegos
    JOIN desarrolladores USING (id_desarrollador)
    WHERE plataforma = 'Switch';
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(b3)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** `JOIN ... USING (columna)` es un atajo para cuando la columna de unión se llama igual en ambas tablas: evita repetir `j.id_desarrollador = d.id_desarrollador` y fusiona esa columna en una sola.

### Ejercicio B4
Muestra el nombre del comprador, el título del juego y el nombre del desarrollador, para todas las compras de juegos del desarrollador `'CD Projekt Red'`.

In [ ]:
b4 = """
    -- escribe tu query aquí
"""

# conn = sqlite3.connect("tienda.db")
# cursor = conn.cursor()
# cursor.execute(b4)
# resultados = cursor.fetchall()
# conn.close()
# resultados

#### Solución B4

In [ ]:
b4 = """
    SELECT c.nombre_comprador, j.titulo, d.nombre
    FROM compras AS c
    JOIN juegos AS j ON c.id_juego = j.id_juego
    JOIN desarrolladores AS d ON j.id_desarrollador = d.id_desarrollador
    WHERE d.nombre = 'CD Projekt Red';
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(b4)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** Para unir 3 tablas encadenamos dos `JOIN`: primero `compras` con `juegos` por `id_juego`, y el resultado con `desarrolladores` por `id_desarrollador`. El `WHERE` final filtra sobre todo lo ya unido.

---
## Bloque C — Subconsultas anidadas

Vimos 4 formas de anidar consultas: en el `FROM`, con `IN` / `NOT IN`, con `EXISTS` / `NOT EXISTS`, y "por valor" (comparando con un solo resultado escalar).

### Ejercicio C1 (resuelto)
Muestra el título de los juegos que cuestan **más** que `'Stardew Valley'`.

In [ ]:
c1 = """
    SELECT titulo
    FROM juegos
    WHERE precio > (SELECT precio FROM juegos WHERE titulo = 'Stardew Valley');
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(c1)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** La subconsulta `(SELECT precio FROM juegos WHERE titulo = 'Stardew Valley')` debe devolver un único valor para poder compararlo con `>`. Si devolviera más de una fila, SQLite lanzaría un error.

### Ejercicio C2 (resuelto — función parametrizada · Ejercicio 7 en las slides)
Escribe una función `compradores_de_desarrollador(db_path, nombre_desarrollador)` que retorne, sin duplicados, los nombres de los compradores que compraron algún juego de un desarrollador dado. Pruébala con `'Nintendo'`.

In [ ]:
def compradores_de_desarrollador(db_path, nombre_desarrollador):
    """Retorna (nombre_comprador,) de los compradores de juegos de un desarrollador dado."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
        SELECT DISTINCT nombre_comprador
        FROM compras
        WHERE id_juego IN (
            SELECT id_juego FROM juegos
            WHERE id_desarrollador = (
                SELECT id_desarrollador FROM desarrolladores WHERE nombre = ?
            )
        );
    """
    cursor.execute(query, (nombre_desarrollador,))
    resultados = cursor.fetchall()
    conn.close()

    return resultados

compradores_de_desarrollador("tienda.db", "Nintendo")

**Explicación:** aunque hay tres niveles de subconsulta anidada, solo un valor viene desde afuera de la query (`nombre_desarrollador`), así que basta un solo `?`, usado en la subconsulta más interna.

### Ejercicio C3 (resuelto)
Muestra el nombre de los desarrolladores que tienen **al menos un** juego con precio menor a 15 dólares.

In [ ]:
c3 = """
    SELECT nombre
    FROM desarrolladores AS d
    WHERE EXISTS (
        SELECT * FROM juegos AS j
        WHERE j.id_desarrollador = d.id_desarrollador AND j.precio < 15
    );
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(c3)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** La subconsulta está *correlacionada*: `j.id_desarrollador = d.id_desarrollador` la conecta con cada fila externa de `desarrolladores`. `EXISTS` solo pregunta si hay *al menos una* fila que cumpla, sin importar cuántas.

### Ejercicio C4
Muestra el nombre de los compradores (sin duplicados) que **no** han comprado ningún juego de la plataforma `'PS5'`.

*Pista: parte de `compras AS c1` y usa `NOT EXISTS` con una subconsulta que junte `compras AS c2` con `juegos`, correlacionando por `nombre_comprador`*

In [ ]:
c4 = """
    -- escribe tu query aquí
"""

# conn = sqlite3.connect("tienda.db")
# cursor = conn.cursor()
# cursor.execute(c4)
# resultados = cursor.fetchall()
# conn.close()
# resultados

#### Solución C4

In [ ]:
c4 = """
    SELECT DISTINCT c1.nombre_comprador
    FROM compras AS c1
    WHERE NOT EXISTS (
        SELECT *
        FROM compras AS c2
        JOIN juegos AS j ON c2.id_juego = j.id_juego
        WHERE c2.nombre_comprador = c1.nombre_comprador AND j.plataforma = 'PS5'
    );
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(c4)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** Es la versión \"negada\" del ejercicio anterior: por cada comprador (`c1`), preguntamos si *no* existe ninguna compra suya (`c2`) de un juego en PS5. Partimos de `compras` (no de una lista de nombres) porque ahí es donde están los nombres de los compradores.

### Ejercicio C5 (resuelto)
Usando una subconsulta en el `FROM`, muestra el título y la plataforma de los juegos \"económicos\" (precio menor a 20), junto con el nombre de su desarrollador.

In [ ]:
c5 = """
    SELECT baratos.titulo, baratos.plataforma, d.nombre
    FROM (SELECT * FROM juegos WHERE precio < 20) AS baratos
    JOIN desarrolladores AS d ON baratos.id_desarrollador = d.id_desarrollador;
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(c5)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** Una subconsulta en el `FROM` se comporta como una tabla más: aquí `baratos` es el resultado de filtrar `juegos` por precio, y luego lo unimos con `desarrolladores` como si fuera cualquier otra tabla.

---
## Bloque D — Operadores de conjunto

`UNION`, `UNION ALL`, `INTERSECT` y `EXCEPT` combinan los resultados de dos queries que tengan las **mismas columnas** (misma cantidad y tipos compatibles).

### Ejercicio D1 (resuelto)
Muestra, sin duplicados, todas las plataformas en las que hay juegos de género `'RPG'` o de género `'Acción'`.

In [ ]:
d1 = """
    SELECT plataforma FROM juegos WHERE genero = 'RPG'
    UNION
    SELECT plataforma FROM juegos WHERE genero = 'Acción';
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(d1)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** `UNION` junta los resultados de ambos `SELECT` y elimina duplicados. Por eso, aunque hay muchos juegos de RPG y de Acción en PC, `'PC'` aparece una sola vez.

### Ejercicio D2 (resuelto)
La misma consulta anterior, pero con `UNION ALL`: ¿qué cambia?

In [ ]:
d2 = """
    SELECT plataforma FROM juegos WHERE genero = 'RPG'
    UNION ALL
    SELECT plataforma FROM juegos WHERE genero = 'Acción';
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(d2)
resultado = cursor.fetchall()
conn.close()

print(f"Filas con UNION ALL: {len(resultado)} (UNION sin ALL solo tenía 2, porque elimina duplicados)")
resultado

**Explicación:** `UNION ALL` no elimina duplicados: por eso el número de filas es exactamente la suma de las filas de ambos `SELECT` por separado.

### Ejercicio D3 (función parametrizada)
Escribe una función `compradores_de_dos_plataformas(db_path, plataforma1, plataforma2)` que retorne los nombres de los compradores que compraron tanto un juego de `plataforma1` como uno de `plataforma2`. Pruébala con `'PC'` y `'Switch'`.

In [ ]:
def compradores_de_dos_plataformas(db_path, plataforma1, plataforma2):
    """Retorna (nombre_comprador,) de compradores que compraron juegos de ambas plataformas."""
    # escribe tu código aquí
    pass

# compradores_de_dos_plataformas("tienda.db", "PC", "Switch")

#### Solución D3

In [ ]:
def compradores_de_dos_plataformas(db_path, plataforma1, plataforma2):
    """Retorna (nombre_comprador,) de compradores que compraron juegos de ambas plataformas."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
        SELECT c.nombre_comprador
        FROM compras AS c JOIN juegos AS j ON c.id_juego = j.id_juego
        WHERE j.plataforma = ?
        INTERSECT
        SELECT c.nombre_comprador
        FROM compras AS c JOIN juegos AS j ON c.id_juego = j.id_juego
        WHERE j.plataforma = ?;
    """
    cursor.execute(query, (plataforma1, plataforma2))
    resultados = cursor.fetchall()
    conn.close()

    return resultados

compradores_de_dos_plataformas("tienda.db", "PC", "Switch")

**Explicación:** dos `?`, uno por cada plataforma. El primero se reemplaza por `plataforma1`, el segundo por `plataforma2`, en el mismo orden en que aparecen en la tupla `(plataforma1, plataforma2)`.

### Ejercicio D4
Usando `EXCEPT`, obtén el título de todos los juegos que nunca han sido comprados.

In [ ]:
d4 = """
    -- escribe tu query aquí
"""

# conn = sqlite3.connect("tienda.db")
# cursor = conn.cursor()
# cursor.execute(d4)
# resultados = cursor.fetchall()
# conn.close()
# resultados

#### Solución D4

In [ ]:
d4 = """
    SELECT titulo FROM juegos
    EXCEPT
    SELECT j.titulo
    FROM juegos AS j JOIN compras AS c ON j.id_juego = c.id_juego;
"""

conn = sqlite3.connect("tienda.db")
cursor = conn.cursor()
cursor.execute(d4)
resultados = cursor.fetchall()
conn.close()

resultados

**Explicación:** `EXCEPT` se queda con las filas del primer `SELECT` que **no** aparecen en el segundo: acá, todos los títulos menos los que sí tienen alguna compra asociada.

---

## Ultimo Ejemplo

In [ ]:
def buscar_juegos(db_path, genero, precio_maximo):
    """
    Retorna (titulo, plataforma, precio) de los juegos de un género dado
    cuyo precio no supere precio_maximo.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
        SELECT titulo, plataforma, precio
        FROM juegos
        WHERE genero = ? AND precio <= ?;
    """
    cursor.execute(query, (genero, precio_maximo))
    resultados = cursor.fetchall()
    conn.close()

    return resultados

buscar_juegos("tienda.db", "RPG", 40)

Nota cómo:
- La query usa `?` como *placeholder* en vez de meter los valores directamente en el string (más seguro, y evita problemas con comillas/tipos).
- `cursor.execute(query, (genero, precio_maximo))` pasa los valores en el mismo orden que aparecen los `?`.
- `cursor.fetchall()` devuelve **todas** las filas como una lista de tuplas.

---

### Referencias
- [Documentación de `sqlite3` en Python](https://docs.python.org/3/library/sqlite3.html)
- [Documentación oficial de SQLite](https://www.sqlite.org/docs.html)